In [9]:
import pandas as pd
import numpy as np

In [10]:
CLINVAR_PATH = "../data/processed/clinvar_filtered.csv"
MUTATION_FEATURES_PATH = "../data/processed/gene_mutation_features.csv"

OUTPUT_PATH = "../data/processed/merged_gene_mutation_features.csv"

In [11]:
clinvar = pd.read_csv(CLINVAR_PATH)
mutation_features = pd.read_csv(MUTATION_FEATURES_PATH)

print("ClinVar shape:", clinvar.shape)
print("Mutation features shape:", mutation_features.shape)

ClinVar shape: (1591701, 8)
Mutation features shape: (23050, 4)


In [12]:
clinvar.columns = clinvar.columns.str.strip()
mutation_features.columns = mutation_features.columns.str.strip()

print(clinvar.columns)
print(mutation_features.columns)

Index(['VariationID', 'GeneSymbol', 'Chromosome', 'Start', 'ReferenceAllele',
       'AlternateAllele', 'ClinicalSignificance', 'Type'],
      dtype='object')
Index(['GeneSymbol', 'total_variants', 'pathogenic_variants',
       'benign_variants'],
      dtype='object')


In [13]:
clinvar["GeneSymbol"] = clinvar["GeneSymbol"].str.upper()
mutation_features["GeneSymbol"] = mutation_features["GeneSymbol"].str.upper()

clinvar = clinvar.dropna(subset=["GeneSymbol"])

In [14]:
variant_type_diversity = (
    clinvar.groupby("GeneSymbol")["Type"]
    .nunique()
    .reset_index()
)

variant_type_diversity.columns = [
    "GeneSymbol",
    "variant_type_diversity"
]

variant_type_diversity.head()

,GeneSymbol,variant_type_diversity
0,-,9
1,A-GAMMA3'E;BGLT3;HBE1;HBG1;HBG2;HS-E1;LOC10609...,1
2,A1BG,1
3,A1CF,1
4,A2M,2


In [15]:
chromosome_diversity = (
    clinvar.groupby("GeneSymbol")["Chromosome"]
    .nunique()
    .reset_index()
)

chromosome_diversity.columns = [
    "GeneSymbol",
    "chromosome_diversity"
]

chromosome_diversity.head()

,GeneSymbol,chromosome_diversity
0,-,24
1,A-GAMMA3'E;BGLT3;HBE1;HBG1;HBG2;HS-E1;LOC10609...,1
2,A1BG,1
3,A1CF,1
4,A2M,1


In [16]:
unique_variant_count = (
    clinvar.groupby("GeneSymbol")["VariationID"]
    .nunique()
    .reset_index()
)

unique_variant_count.columns = [
    "GeneSymbol",
    "unique_variant_count"
]

unique_variant_count.head()

,GeneSymbol,unique_variant_count
0,-,728
1,A-GAMMA3'E;BGLT3;HBE1;HBG1;HBG2;HS-E1;LOC10609...,1
2,A1BG,2
3,A1CF,6
4,A2M,40


In [17]:
variant_features = variant_type_diversity.merge(
    chromosome_diversity,
    on="GeneSymbol",
    how="outer"
)

variant_features = variant_features.merge(
    unique_variant_count,
    on="GeneSymbol",
    how="outer"
)

variant_features.fillna(0, inplace=True)

variant_features.head()

,GeneSymbol,variant_type_diversity,chromosome_diversity,unique_variant_count
0,-,9,24,728
1,A-GAMMA3'E;BGLT3;HBE1;HBG1;HBG2;HS-E1;LOC10609...,1,1,1
2,A1BG,1,1,2
3,A1CF,1,1,6
4,A2M,2,1,40


In [18]:
mutation_features = mutation_features.drop_duplicates(subset=["GeneSymbol"])
variant_features = variant_features.drop_duplicates(subset=["GeneSymbol"])

In [19]:
merged_features = mutation_features.merge(
    variant_features,
    on="GeneSymbol",
    how="left"
)

merged_features.fillna(0, inplace=True)

merged_features.head()

,GeneSymbol,total_variants,pathogenic_variants,benign_variants,variant_type_diversity,chromosome_diversity,unique_variant_count
0,-,737,34,420,9,24,728
1,A-GAMMA3'E;BGLT3;HBE1;HBG1;HBG2;HS-E1;LOC10609...,1,1,0,1,1,1
2,A1BG,2,0,0,1,1,2
3,A1CF,6,0,4,1,1,6
4,A2M,40,0,18,2,1,40


In [20]:
merged_features = merged_features.loc[
    :, ~merged_features.columns.duplicated()
]

In [21]:
print("Total genes:", merged_features.shape[0])

print(
    "Duplicate genes:",
    merged_features["GeneSymbol"].duplicated().sum()
)

Total genes: 23050
Duplicate genes: 0


In [22]:
merged_features.to_csv(
    OUTPUT_PATH,
    index=False
)

print("Merged dataset saved successfully")

Merged dataset saved successfully
